# Axis-Aligned Bounding Box (AABB) Minimization

For a set of objects in space, their bounding volume is 
a closed region that fully contains all objects in the set.
The axis-aligned bounding box is the simplest form of a bounding volume,
having the restriction that the volume must be a rectangular prism
where all edges are parallel to one of the coordinate system axes.

The complexity of many computer-aided drug design algorithms (e.g., grid-based calculations)
is directly proportional to the AABB volume.
Since in most cases, the absolute position of a chemical system does not contain any meaningful information,
one can simply rotate the entire system such that the AABB volume is minimized,
thus reducing the workload of all downstream steps that depend on the AABB volume.

Entries in the Protein Data Bank (PDB) commonly do not have minimized AABB volumes,
thus using them directly in AABB-volume-dependent algorithms can result in a significant
increase in calculation time.

To minimize the AABB of a chemical system,
one can simply call the `minimize_aabb()` method of a `ChemicalSystem` object.

In [1]:
import numpy as np
import caddpy
import sciapi
import scids
import scishow

## Example

Create a chemical system, e.g., from a PDB ID:

In [2]:
chemsys_before = caddpy.chemsys.from_pdb(sciapi.pdb.file.entry("6ANO", "pdb"))

Call its `minimize_aabb()` method to get an identical `ChemicalSystem` object where the AABB is minimized:

In [3]:
chemsys_after = chemsys_before.minimize_aabb()

In [4]:
assert chemsys_before.composition == chemsys_after.composition
assert chemsys_before.trajectory.points.shape == chemsys_after.trajectory.points.shape

### Results

AABB volume of the original system:

In [5]:
aabb_before = chemsys_before.trajectory.aabb()
aabb_before

RectangularCuboid(
    lower_bounds=[ -0.826 -54.541 -17.661],
    upper_bounds=[36.679 40.24  54.777],
    batch=0,
)

In [6]:
volume_before = aabb_before.volume
volume_before

Array(257499.84, dtype=float32)

AABB volume of the minimized system:

In [7]:
aabb_after = chemsys_after.trajectory.aabb()
aabb_after

RectangularCuboid(
    lower_bounds=[-43.79601  -28.297985 -25.326345],
    upper_bounds=[66.233116  -6.4053526 -2.5143282],
    batch=0,
)

In [8]:
volume_after = aabb_after.volume
volume_after

Array(54950.207, dtype=float32)

Volume reduction in percent:

In [9]:
reduction = 1 - (volume_after / volume_before)
print(f"{reduction * 100:.2f}")

78.66


Visualization of both AABBs:

In [10]:
nw=scishow.nglview.NGLWidget()
nw.add_trajectory(chemsys_before, name="before")
nw.add_trajectory(chemsys_after, name="after")
nw.add_origin()
nw.add_bounding_box(0, name="bbox_before")
nw.add_bounding_box(1, name="bbox_after")
nw.display(gui=True)

ThemeManager()

NGLWidget(gui_style='ngl')

## PDB Statistics
Here we randomly select a number of PDB entries and calculate their AABB volume reduction after minimization:

In [11]:
SAMPLE_SIZE = 100

In [12]:
all_pdb_ids = sciapi.pdb.data.holdings()
samples = []
exceptions = []
for pdb_id_idx in np.random.permutation(len(all_pdb_ids)):
    n_cases = len(samples)
    if n_cases == SAMPLE_SIZE:
        break
    pdb_id = all_pdb_ids[pdb_id_idx]
    pdb_data = sciapi.pdb.data.entry(pdb_id)
    if len(pdb_data["rcsb_entry_container_identifiers"]["model_ids"]) > 1:
        # Skip multi-model entries; since they first need to be aligned
        continue
    try:
        pdbfile = sciapi.pdb.file.entry(pdb_id, "pdb")
    except Exception as e:
        # Not all entries have PDB versions
        print(f"  --  {pdb_id}: Entry without PDB file.")
        continue

    try:
        chemsys_before = caddpy.chemsys.from_pdb(pdbfile)
    except Exception as e:
        # Some PDB enties may have invalid data,
        # e.g. entry '6D0S' has atoms with element symbol 'X'
        print(f"  --  {pdb_id}: Failed to parse PDB file.")
        exceptions.append((pdb_id, e))
        continue
    chemsys_after = chemsys_before.minimize_aabb()
    aabb_before = chemsys_before.trajectory.aabb()
    aabb_after = chemsys_after.trajectory.aabb()
    volume_before = aabb_before.volume
    volume_after = aabb_after.volume
    reduction = 1 - (volume_after / volume_before)
    samples.append(
        {
            "pdb_id": pdb_id,
            "chemsys_before": chemsys_before,
            "chemsys_pca": chemsys_after,
            "reduction": reduction
        }
    )
    print(f"{n_cases + 1:>4}. {pdb_id}: {reduction * 100:.2f}%")

   1. 2R6V: 27.31%
   2. 3ERS: 24.49%
   3. 7QJ2: 23.71%
   4. 3V7T: 6.86%
   5. 8S5M: 0.40%
   6. 2CAD: 26.41%
   7. 4DBD: 20.59%
   8. 1D4M: 23.80%
   9. 7HLT: 33.60%
  10. 3TIO: 22.77%
  11. 6W0E: 45.76%
  12. 4UBG: 28.39%
  13. 2AY3: 12.18%
  14. 6CCB: 36.06%


/Volumes/T7/repo/gh/aariam/opencadd/scifile/pkg/src/scifile/pdb/parser.py:143: UserWarning: Error parsing record atom: could not convert string to float: np.str_('\x000')
  warnings.warn(


  --  8W7R: Failed to parse PDB file.
  15. 1PRP: 24.88%
  16. 3LPD: 31.20%
  17. 4KWC: 1.32%
  18. 4PLJ: 50.84%
  19. 3B5W: 36.12%
  20. 7ZX2: 35.79%
  21. 9GNL: 28.55%
  22. 1K21: 9.68%
  23. 3DAI: 24.71%
  24. 8CKO: 16.12%


/Volumes/T7/repo/gh/aariam/opencadd/scifile/pkg/src/scifile/pdb/parser.py:143: UserWarning: Error parsing record atom: could not convert string to float: np.str_('\x000')
  warnings.warn(


  --  7L70: Failed to parse PDB file.
  25. 6FNS: 10.37%
  26. 6TUS: 30.84%
  27. 8Q27: 60.21%
  28. 1MUC: 18.43%
  --  7QXD: Entry without PDB file.
  29. 4QOM: 30.30%
  30. 8DRT: 8.15%
  31. 7C2E: 23.12%
  32. 6TOT: 23.79%
  33. 5RWZ: 36.88%
  34. 6XMU: 7.56%
  35. 3BNH: 8.81%
  36. 3FFG: 3.29%
  37. 6CDA: 27.40%
  38. 1WL9: 27.97%
  39. 6X9C: 11.76%
  40. 3M8L: 13.78%
  41. 7KE9: 15.17%
  42. 5XS9: 7.08%
  43. 8EHD: 12.76%
  44. 9J12: 8.79%
  45. 8JF5: 45.21%
  46. 1L72: 10.11%
  47. 7WIG: 14.74%
  --  7YIW: Entry without PDB file.
  48. 3N0R: 24.91%
  49. 5Q7A: 8.14%
  50. 4BRB: 38.97%
  51. 4ARM: 1.06%
  52. 5NEU: 15.21%
  53. 5HNO: 15.83%
  54. 8QK6: 2.23%
  55. 1WV9: 33.40%
  56. 7PA0: 14.22%
  57. 2EBY: 8.02%
  58. 5Q2J: 3.89%
  59. 8EZ7: 37.52%
  60. 1NUV: 46.52%
  61. 2CAL: 20.62%
  62. 2PWS: 31.04%
  63. 6ZMY: 21.57%
  64. 5EI4: 32.08%
  65. 3O9T: 36.62%
  66. 5QR1: 4.84%
  67. 3UP3: 21.50%
  68. 5TN9: 0.00%
  69. 6NQT: 28.08%
  70. 4GJQ: 9.94%
  71. 4UYM: 13.85%
  72. 3UBQ:

In [13]:
reductions = [sample["reduction"] * 100 for sample in samples]

Maximum volume reduction:

In [14]:
print(f"{np.max(reductions):.2f}")

60.21


Average volume reduction:

In [15]:
print(f"{np.mean(reductions):.2f}")

22.27
